# Database APIs

> Monitoring data will be stored in a SQLlite DB. This module exposes all necessary APIs for the same.

In [ ]:
#| default_exp db

In [ ]:
#| hide
from nbdev.showdoc import *

The process monitor is detecting events with timestamps.  

The next logical step is to **persist these events** rather than just printing them. That means building the **storage module** (`db.py`) with SQLite.
2 options in my mind  
1. store as events - pid, app, start/ stop, timestamp (only inserts)  
2. create new record at open - pid, app, start time, update stop time on stop based on pid and not sttopped (based on NULL stop time)  

**Option 1 (event log)** — Simpler to write (inserts only), but querying "how long was Chrome open?" requires matching start/stop pairs afterwards.  
**Option 2 (session record)** — Slightly more complex (insert + update), but gives you duration directly. The risk: if the app or machine crashes, you'll have orphan rows with NULL stop times.  

**Option 1 has less locking risk** — it's insert-only, so each write is a quick, independent transaction. Option 2 requires a SELECT to find the matching row, then an UPDATE, which holds the lock longer.  
  

Process monitor output

```
Opened processes: 2026-02-21 16:20:08-{(874, 'sleep')}
Closed processes: 2026-02-21 16:20:23-{(870, 'top')}
Opened processes: 2026-02-21 16:21:08-{(875, 'top')}
Closed processes: 2026-02-21 16:21:23-{(875, 'top')}
```

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from fastlite import *
import snooper_pkg.config as cf

In [ ]:
#| exporti
db = database(cf.DATABASE_PATH)

In [ ]:
#| export
class Event:
    """The class container for process monitoring"""
    pid: int # process identifier
    app: str # application name
    event_type: str # start of stop
    timestamp: str # time of event
    id: int = None # auto generated primary key

In [ ]:
#| hide
e = db.create(Event, transform = True)
e

<Table event (pid, app, event_type, timestamp, id)>

In [ ]:
#| hide
e.insert(pid=874, app='sleep', event_type='start', timestamp='2026-02-21 16:10:08')

Event(pid=874, app='sleep', event_type='start', timestamp='2026-02-21 16:10:08', id=11)

In [ ]:
#| export
def log_event(
    pid: int, # process identifier
    app: str, # application name
    event_type: str, # start of stop
    timestamp: str, # time of event
    ):
    """Creates the database if it does not exist and inserts a record"""
    events = db.create(Event, transform = True)
    e = events.insert(pid=pid, app=app, event_type=event_type, timestamp=timestamp)
    return e

In [ ]:
log_event(pid=874, app='sleep', event_type='stop', timestamp='2026-02-21 16:20:38')

Event(pid=874, app='sleep', event_type='stop', timestamp='2026-02-21 16:20:38', id=12)

In [ ]:
#| hide
db.t

event

In [ ]:
#| hide
e(limit = 5)

[Event(pid=874, app='sleep', event_type='start', timestamp='2026-02-21 16:10:08', id=1),
 Event(pid=874, app='sleep', event_type='stop', timestamp='2026-02-21 16:20:38', id=2),
 Event(pid=874, app='sleep', event_type='start', timestamp='2026-02-21 16:10:08', id=3),
 Event(pid=874, app='sleep', event_type='stop', timestamp='2026-02-21 16:20:38', id=4),
 Event(pid=874, app='sleep', event_type='start', timestamp='2026-02-21 16:10:08', id=5)]

In [ ]:
#| hide
db.q("""
SELECT 
    s.pid, 
    s.app, 
    s.timestamp AS start_time, 
    e.timestamp AS stop_time,
    e.timestamp-s.timestamp as duration
FROM event s 
JOIN event e 
    ON s.pid = e.pid 
    AND s.app = e.app
WHERE s.event_type = 'start' 
    AND e.event_type = 'stop'
""")

[{'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration': 0},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration': 0},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration': 0},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration': 0},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration': 0},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration': 0},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration': 0},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  

In [ ]:
#| hide
db.q("""
SELECT 
    s.pid, 
    s.app, 
    s.timestamp AS start_time, 
    e.timestamp AS stop_time,
    (julianday(e.timestamp) - julianday(s.timestamp)) * 86400 AS duration_seconds
FROM event s 
JOIN event e 
    ON s.pid = e.pid 
    AND s.app = e.app
WHERE s.event_type = 'start' 
    AND e.event_type = 'stop'
""")

[{'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16

In [ ]:
#| export
def get_processes():
    """Returns application usage with start time, stop time and duration in seconds"""
    return db.q("""
            SELECT 
                s.pid, 
                s.app, 
                s.timestamp AS start_time, 
                e.timestamp AS stop_time,
                (julianday(e.timestamp) - julianday(s.timestamp)) * 86400 AS duration_seconds
            FROM event s 
            JOIN event e 
                ON s.pid = e.pid 
                AND s.app = e.app
            WHERE s.event_type = 'start' 
                AND e.event_type = 'stop'
            """)

In [ ]:
get_processes()

[{'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16:20:38',
  'duration_seconds': 630.000002682209},
 {'pid': 874,
  'app': 'sleep',
  'start_time': '2026-02-21 16:10:08',
  'stop_time': '2026-02-21 16

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()